**Run evidence** · task `gold_supplier` · run `403873149252985` · result **SUCCESS** · executed 2026-10-19 17:27 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `02_build_gold/02b_supplier`.

# 02b — Workflow B → SUPPLIER (grain: LIFNR + EKORG)

Consolidates vendor master into one row per vendor + purchasing org.

**Steps (spec §4, Workflow B):**
1. Start from `RAW_LFM1` (already at LIFNR+EKORG grain; carries SPERM block flag).
2. Left-join `RAW_LFA1` on LIFNR, broadcasting NAME1/LAND1 to every EKORG row.
3. **Drop `RAW_LFB1`** (company-code grain) — not needed for purchasing questions.
4. Output: one row per LIFNR+EKORG with NAME1, LAND1, SPERM.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
lfm1 = spark.table(CFG.fq("RAW_LFM1"))   # LIFNR, EKORG, SPERM  (target grain)
lfa1 = spark.table(CFG.fq("RAW_LFA1"))   # LIFNR, NAME1, LAND1

supplier = (
    lfm1
    .join(lfa1, on="LIFNR", how="left")   # broadcast NAME1/LAND1 to every EKORG row
    .select("LIFNR", "EKORG", "NAME1", "LAND1", "SPERM")
)
# Note: RAW_LFB1 (company-code grain) is intentionally NOT joined (spec).

(supplier.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(CFG.fq("SUPPLIER")))

cnt = spark.table(CFG.fq("SUPPLIER")).count()
dupes = (spark.table(CFG.fq("SUPPLIER"))
         .groupBy("LIFNR", "EKORG").count().filter("count > 1").count())
print(f"SUPPLIER rows: {cnt:,} | grain violations (LIFNR+EKORG): {dupes}")
assert dupes == 0, "SUPPLIER grain violated — LIFNR+EKORG not unique!"
display(spark.table(CFG.fq("SUPPLIER")).limit(10))

SUPPLIER rows: 42 | grain violations (LIFNR+EKORG): 0


LIFNR,EKORG,NAME1,LAND1,SPERM
V200000,2000,Vendor 00 GmbH,PL,
V200000,1000,Vendor 00 GmbH,PL,
V200001,2000,Supplier 01 Ltd,DE,
V200002,2000,Vendor 02 GmbH,DE,
V200002,1000,Vendor 02 GmbH,DE,
V200003,2000,Supplier 03 Ltd,PL,
V200003,1000,Supplier 03 Ltd,PL,
V200004,2000,Vendor 04 GmbH,IT,
V200005,2000,Supplier 05 Ltd,FR,
V200005,1000,Supplier 05 Ltd,FR,


In [ ]:
dbutils.notebook.exit(f"02b_supplier: SUCCESS — {cnt} rows.")

Notebook exited: 02b_supplier: SUCCESS — 42 rows.
